# 🔒 Day 01a: SSL/TLS — Handshake, Certificates & HTTPS

---

## 🎯 What You'll Master Today
- TLS 1.2 & 1.3 handshake step-by-step
- Certificates and CA chain of trust
- Symmetric vs asymmetric key exchange
- HTTPS = HTTP + TLS

---

### 🎭 The Analogy

TLS handshake = two strangers meeting at a coffee shop:  
1. Show your **ID card** (certificate) to prove who you are  
2. Agree on a **secret language** only you two understand (session key)  
3. Now talk freely — nobody else can eavesdrop

```
╔══════════════════════════════════════════════════════════════════════╗
║                    TLS 1.2 HANDSHAKE                                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Client                                          Server              ║
║    │                                                │                ║
║    │──── ClientHello (ciphers, random) ──────────→│  1. Client says║
║    │                                                │     hello      ║
║    │←─── ServerHello + Certificate + Done ────────│  2. Server     ║
║    │                                                │     responds  ║
║    │──── Key Exchange + ChangeCipherSpec ────────→│  3. Key setup  ║
║    │                                                │                ║
║    │←─── ChangeCipherSpec + Finished ─────────────│  4. Done!      ║
║    │                                                │                ║
║    │═══════ Encrypted Application Data ══════════│  5. Secure     ║
║    │                                                │     channel   ║
║                                                                      ║
║    2 round trips (TLS 1.2) vs 1 round trip (TLS 1.3)              ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. TLS Handshake Simulator
# ============================================
import hashlib
import os

class TLSHandshake:
    def __init__(self, version='1.2'):
        self.version = version
        self.client_random = None
        self.server_random = None
        self.session_key = None
    
    def simulate(self):
        print(f"  ═══ TLS {self.version} Handshake Simulation ═══\n")
        
        # Step 1: ClientHello
        self.client_random = os.urandom(16).hex()[:16]
        ciphers = ['TLS_AES_256_GCM_SHA384', 'TLS_CHACHA20_POLY1305_SHA256']
        print(f"  ► Step 1: CLIENT HELLO")
        print(f"    Client Random : {self.client_random}")
        print(f"    Supported     : TLS {self.version}")
        print(f"    Cipher Suites : {', '.join(ciphers)}")
        print(f"    SNI           : www.example.com")
        print()
        
        # Step 2: ServerHello + Certificate
        self.server_random = os.urandom(16).hex()[:16]
        cert_fingerprint = hashlib.sha256(b'example.com cert').hexdigest()[:32]
        print(f"  ► Step 2: SERVER HELLO + CERTIFICATE")
        print(f"    Server Random : {self.server_random}")
        print(f"    Selected Cipher: {ciphers[0]}")
        print(f"    Certificate    : CN=www.example.com")
        print(f"    Cert Fingerprint: {cert_fingerprint}")
        print(f"    Issuer         : Let's Encrypt Authority X3")
        print(f"    CA Chain       : Root CA → Intermediate CA → Server Cert")
        print()
        
        # Step 3: Key Exchange
        premaster = hashlib.sha256(
            (self.client_random + self.server_random).encode()
        ).hexdigest()[:32]
        self.session_key = hashlib.sha256(
            (premaster + self.client_random + self.server_random).encode()
        ).hexdigest()[:32]
        
        print(f"  ► Step 3: KEY EXCHANGE")
        print(f"    Pre-master Secret: {premaster} (encrypted with server's public key)")
        print(f"    Master Secret    : derived from pre-master + randoms")
        print(f"    Session Key      : {self.session_key}")
        print()
        
        # Step 4: Finished
        print(f"  ► Step 4: CHANGE CIPHER SPEC + FINISHED")
        print(f"    Both sides now use symmetric encryption")
        print(f"    All future data encrypted with session key")
        print()
        
        if self.version == '1.3':
            print(f"  TLS 1.3 Improvements:")
            print(f"    • 1-RTT handshake (was 2-RTT in 1.2)")
            print(f"    • 0-RTT resumption for repeat connections")
            print(f"    • Removed weak ciphers (RSA key exchange, CBC)")
            print(f"    • All handshake messages encrypted after ServerHello")
        
        return self.session_key

tls = TLSHandshake('1.2')
key = tls.simulate()

In [ ]:
# ============================================
# 2. Certificate Chain of Trust
# ============================================

class Certificate:
    def __init__(self, subject, issuer, is_ca=False):
        self.subject = subject
        self.issuer = issuer
        self.is_ca = is_ca
        self.fingerprint = hashlib.sha256(f"{subject}{issuer}".encode()).hexdigest()[:16]

def verify_chain(certs):
    print("  ═══ Certificate Chain Verification ═══\n")
    
    for i, cert in enumerate(certs):
        indent = "    " + "  " * i
        role = "[Root CA]" if i == 0 else "[Intermediate CA]" if cert.is_ca else "[Server Cert]"
        print(f"{indent}┌─ {role}")
        print(f"{indent}│  Subject : {cert.subject}")
        print(f"{indent}│  Issuer  : {cert.issuer}")
        print(f"{indent}│  SHA-256 : {cert.fingerprint}")
        
        if i == 0:
            if cert.subject == cert.issuer:
                print(f"{indent}│  ✅ Self-signed root (trusted by OS/browser)")
            else:
                print(f"{indent}│  ❌ Root not self-signed!")
        else:
            if cert.issuer == certs[i-1].subject:
                print(f"{indent}│  ✅ Signed by {cert.issuer}")
            else:
                print(f"{indent}│  ❌ Issuer mismatch!")
        print(f"{indent}└─")

chain = [
    Certificate("DigiCert Global Root",      "DigiCert Global Root",      is_ca=True),
    Certificate("DigiCert SHA2 Intermediate", "DigiCert Global Root",      is_ca=True),
    Certificate("www.example.com",             "DigiCert SHA2 Intermediate", is_ca=False),
]

verify_chain(chain)

print("  Trust flow: Browser's trusted root store")
print("             → validates Root CA")
print("             → Root signed Intermediate")
print("             → Intermediate signed Server cert")
print("             → Server cert matches domain ✅")

In [ ]:
# ============================================
# 3. HTTPS Connection Lifecycle
# ============================================

def https_lifecycle():
    steps = [
        ("DNS",          "Resolve www.example.com → 93.184.216.34"),
        ("TCP",          "3-way handshake (SYN → SYN-ACK → ACK)"),
        ("TLS",          "TLS handshake (ClientHello → ... → Finished)"),
        ("HTTP Request", "GET / HTTP/1.1 (encrypted inside TLS)"),
        ("HTTP Response", "200 OK + HTML body (encrypted inside TLS)"),
        ("TLS Close",    "close_notify alert"),
        ("TCP Close",    "4-way FIN handshake"),
    ]
    
    print("  ═══ HTTPS Request Lifecycle ═══\n")
    for i, (phase, desc) in enumerate(steps, 1):
        bar = "█" * (7 - abs(i - 4))  # visual shape
        print(f"    {i}. [{phase:>15}] {bar:>7} {desc}")
    
    print(f"\n  Total round trips:")
    print(f"    DNS:  1 RTT (if not cached)")
    print(f"    TCP:  1 RTT (SYN + SYN-ACK)")
    print(f"    TLS:  1-2 RTT (1.3 vs 1.2)")
    print(f"    HTTP: 1 RTT (request + response)")
    print(f"    ─────────────────────")
    print(f"    Total: 4-5 RTTs for first request")
    print(f"           (subsequent: TCP+TLS reuse → 1 RTT)")

https_lifecycle()

In [ ]:
# ============================================
# 4. TLS 1.2 vs 1.3 Comparison
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════╗
  ║                    TLS 1.2 vs TLS 1.3                               ║
  ╠═════════════════════╤═══════════════════╤═══════════════════════════╣
  ║ Feature             │ TLS 1.2           │ TLS 1.3                   ║
  ╠═════════════════════╪═══════════════════╪═══════════════════════════╣
  ║ Handshake RTTs      │ 2                 │ 1 (0-RTT for resumption) ║
  ║ Key Exchange        │ RSA, DHE, ECDHE   │ ECDHE only (forward sec) ║
  ║ Cipher Suites       │ 37 supported      │ 5 supported (only safe)  ║
  ║ Handshake encrypted │ No (plaintext)    │ Yes (after ServerHello)  ║
  ║ Forward Secrecy     │ Optional (if DHE) │ Mandatory               ║
  ║ 0-RTT Resumption    │ No                │ Yes (PSK mode)           ║
  ║ Removed             │ N/A               │ RSA key exchange, RC4,   ║
  ║                     │                   │ CBC mode, SHA-1, MD5     ║
  ╚═════════════════════╧═══════════════════╧═══════════════════════════╝

  Key Concepts:
  • Forward Secrecy: even if server's private key is later compromised,
    past sessions can't be decrypted (each session has unique DH keys)
  • SNI (Server Name Indication): client tells server which domain it
    wants (needed for virtual hosting with TLS)
  • HSTS: HTTP Strict Transport Security — forces HTTPS-only
  • Certificate Pinning: app hardcodes expected cert fingerprint
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | TLS handshake steps? | ClientHello → ServerHello+Cert → Key Exchange → ChangeCipherSpec → Encrypted data |
| 2 | Symmetric vs asymmetric in TLS? | Asymmetric (RSA/ECDH) for key exchange. Symmetric (AES) for data — faster |
| 3 | What is forward secrecy? | Compromise of long-term key doesn't expose past sessions. Achieved via ephemeral DH |
| 4 | TLS 1.2 vs 1.3? | 1.3: 1-RTT, forward secrecy mandatory, weak ciphers removed, handshake encrypted |
| 5 | What does a CA do? | Certificate Authority vouches for server's identity. Signs certs. Browser trusts root CAs |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • TLS = encryption layer between TCP and HTTP           │
## │  • Asymmetric crypto for key exchange, symmetric for data│
## │  • Certificate chain: Root CA → Intermediate → Server    │
## │  • TLS 1.3: faster (1-RTT), safer (forward secrecy)     │
## │  • HTTPS first request: ~4-5 RTTs total                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Encryption & VPN** — Symmetric/asymmetric encryption, hashing, VPN, firewalls